In [3]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
from backtesting.test import SMA, GOOG

In [4]:
#读取文件
bu = pd.read_csv('data/bu.csv')
jd = pd.read_csv('data/jd.csv')
l = pd.read_csv('data/l.csv')
pp = pd.read_csv('data/pp.csv')
ru = pd.read_csv('data/ru.csv')
v = pd.read_csv('data/v.csv')
bu

,date,open,high,low,close,volume,hold,settle
0,2015-06-26,2948.0,2972.0,2936.0,2960.0,32378,50834,2954.0
1,2015-06-29,2948.0,2970.0,2922.0,2936.0,31124,49566,2952.0
2,2015-06-30,2930.0,2948.0,2804.0,2812.0,52186,48312,2854.0
3,2015-07-01,2832.0,2850.0,2808.0,2826.0,31576,49450,2834.0
4,2015-07-02,2826.0,2830.0,2774.0,2816.0,36706,50466,2802.0
...,...,...,...,...,...,...,...,...
2427,2025-06-20,3738.0,3789.0,3723.0,3747.0,253286,284513,3751.0
2428,2025-06-23,3736.0,3804.0,3719.0,3781.0,317501,311332,3769.0
2429,2025-06-24,3762.0,3770.0,3562.0,3580.0,460577,269578,3643.0
2430,2025-06-25,3565.0,3583.0,3537.0,3574.0,282213,251579,3556.0


In [5]:
#重命名各列
def column_rename(df):
    df=df.rename(columns={
        df.columns[0]: 'Date',
        df.columns[1]: 'Open',
        df.columns[2]: 'High',
        df.columns[3]: 'Low',
        df.columns[4]: 'Close',
        df.columns[5]: 'Volume'
    }).drop(columns=[df.columns[6], df.columns[7]])
    df['Date'] = pd.to_datetime(df['Date'])
    return df.set_index('Date')
    
new_bu = column_rename(bu)

new_jd = column_rename(jd)
new_l = column_rename(l)
new_pp = column_rename(pp)
new_ru = column_rename(ru)
new_v = column_rename(v)
print(new_jd.head())
GOOG.head()

              Open    High     Low   Close  Volume
Date                                              
2015-06-26  4145.0  4145.0  4073.0  4085.0   76016
2015-06-29  4065.0  4090.0  4045.0  4068.0   70708
2015-06-30  4060.0  4060.0  3965.0  3969.0   96574
2015-07-01  4000.0  4112.0  3992.0  4028.0   69972
2015-07-02  4026.0  4029.0  3983.0  4009.0   54432


,Open,High,Low,Close,Volume
2004-08-19,100.00,104.06,95.96,100.34,22351900
2004-08-20,101.01,109.08,100.50,108.31,11428600
2004-08-23,110.75,113.48,109.05,109.40,9137200
2004-08-24,111.24,111.60,103.57,104.87,7631300
2004-08-25,104.96,108.00,103.88,106.00,4598900


In [6]:
# 纯双均值策略（0）
class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)

    def next(self):
        if crossover(self.ma1, self.ma2):
            self.buy()
        elif crossover(self.ma2, self.ma1):
            self.sell()

In [7]:
#ATR计算函数
def ATR(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = abs(df['High'] - df['Close'].shift())
    low_close = abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


In [8]:
# 止损卖出（2）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        
        self.entry_price = 0
        self.stop_loss = 0

    def next(self):
        current_close = self.data.Close[-1]
        
        # 修改买入条件：添加持仓状态检查
        if self.entry_price ==0:
            if crossover(self.ma1, self.ma2) :
                self.buy()
                self.entry_price = current_close
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier
        
        # 强化卖出条件：添加持仓状态检查
        elif self.entry_price != 0:
            # 动态更新止损位
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            
            if crossover(self.ma2, self.ma1) or current_close < self.stop_loss:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位

In [52]:
# 尝试提前操作（3）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        
        self.entry_price = 0
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            if self.cross2==self.current_date and below_ma50 and max(self.cross1,self.fall1,self.fall2)==self.cross1 and below_ma50:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
        
        elif self.entry_price != 0:
            # 动态更新止损位
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位
            #if current_close < self.stop_loss or crossover(self.ma2, self.ma1):
            if current_close < self.stop_loss :
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位

In [53]:
bt = Backtest(new_bu, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/2382 [00:00<?, ?bar/s]

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    97.36842
Equity Final [$]                    74510.334
Equity Peak [$]                    181263.968
Commissions [$]                      33962.96
Return [%]                          -25.48967
Buy & Hold Return [%]                52.78731
Return (Ann.) [%]                    -3.00278
Volatility (Ann.) [%]                32.80594
CAGR [%]                             -2.00928
Sharpe Ratio                         -0.09153
Sortino Ratio                         -0.1364
Calmar Ratio                         -0.04889
Alpha [%]                            -8.61004
Beta                                 -0.31977
Max. Drawdown [%]                   -61.41962
Avg. Drawdown [%]                    -9.70341
Max. Drawdown Duration     1914 days 00:00:00
Avg. Drawdown Duration      115 days 00:00:00
# Trades                          

In [54]:
bt = Backtest(new_jd, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    94.40789
Equity Final [$]                    36711.832
Equity Peak [$]                    111299.748
Commissions [$]                     13400.068
Return [%]                          -63.28817
Buy & Hold Return [%]                -9.06762
Return (Ann.) [%]                    -9.86242
Volatility (Ann.) [%]                34.68736
CAGR [%]                             -6.67921
Sharpe Ratio                         -0.28432
Sortino Ratio                        -0.36339
Calmar Ratio                         -0.12736
Alpha [%]                           -67.46628
Beta                                 -0.46077
Max. Drawdown [%]                    -77.4352
Avg. Drawdown [%]                   -20.29831
Max. Drawdown Duration     3221 days 00:00:00
Avg. Drawdown Duration      586 days 00:00:00
# Trades                          

In [55]:
bt = Backtest(new_l, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    96.54463
Equity Final [$]                    37866.858
Equity Peak [$]                     146731.39
Commissions [$]                     21475.842
Return [%]                          -62.13314
Buy & Hold Return [%]               -14.51991
Return (Ann.) [%]                    -9.57637
Volatility (Ann.) [%]                20.89304
CAGR [%]                             -6.47958
Sharpe Ratio                         -0.45835
Sortino Ratio                        -0.57365
Calmar Ratio                         -0.12709
Alpha [%]                           -69.46672
Beta                                 -0.50507
Max. Drawdown [%]                   -75.35166
Avg. Drawdown [%]                    -9.10025
Max. Drawdown Duration     1914 days 00:00:00
Avg. Drawdown Duration      162 days 00:00:00
# Trades                          

In [56]:
bt = Backtest(new_pp, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    93.78856
Equity Final [$]                    128217.89
Equity Peak [$]                    239090.438
Commissions [$]                     36992.378
Return [%]                           28.21789
Buy & Hold Return [%]                -6.59658
Return (Ann.) [%]                     2.61009
Volatility (Ann.) [%]                21.42897
CAGR [%]                              1.72947
Sharpe Ratio                           0.1218
Sortino Ratio                         0.18303
Calmar Ratio                          0.05111
Alpha [%]                            27.46604
Beta                                 -0.11398
Max. Drawdown [%]                   -51.06889
Avg. Drawdown [%]                    -5.14561
Max. Drawdown Duration     2109 days 00:00:00
Avg. Drawdown Duration       84 days 00:00:00
# Trades                          

In [57]:
bt = Backtest(new_ru, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Start                     2015-06-26 00:00:00
End                       2025-06-26 00:00:00
Duration                   3653 days 00:00:00
Exposure Time [%]                    97.61513
Equity Final [$]                      48162.0
Equity Peak [$]                     152933.34
Commissions [$]                      16363.73
Return [%]                            -51.838
Buy & Hold Return [%]                37.91749
Return (Ann.) [%]                     -7.2909
Volatility (Ann.) [%]                73.79314
CAGR [%]                              -4.9151
Sharpe Ratio                          -0.0988
Sortino Ratio                        -0.18304
Calmar Ratio                         -0.07844
Alpha [%]                           -20.57329
Beta                                 -0.82455
Max. Drawdown [%]                    -92.9476
Avg. Drawdown [%]                   -10.86258
Max. Drawdown Duration     1921 days 00:00:00
Avg. Drawdown Duration      155 days 00:00:00
# Trades                          

In [58]:
bt = Backtest(new_v, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/192 [00:00<?, ?bar/s]

Start                     2021-09-15 00:00:00
End                       2022-09-15 00:00:00
Duration                    365 days 00:00:00
Exposure Time [%]                    59.91736
Equity Final [$]                   108445.516
Equity Peak [$]                    117929.728
Commissions [$]                      1533.988
Return [%]                            8.44552
Buy & Hold Return [%]               -20.69393
Return (Ann.) [%]                     8.80945
Volatility (Ann.) [%]                23.20355
CAGR [%]                              5.75733
Sharpe Ratio                          0.37966
Sortino Ratio                         0.63685
Calmar Ratio                          0.57007
Alpha [%]                             8.08663
Beta                                 -0.01734
Max. Drawdown [%]                   -15.45319
Avg. Drawdown [%]                    -4.13036
Max. Drawdown Duration      148 days 00:00:00
Avg. Drawdown Duration       30 days 00:00:00
# Trades                          